In [1]:
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    log_loss,
    roc_auc_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    StratifiedKFold,
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler


DATA_DIR = Path.cwd()

eda6 = pd.read_csv(
    DATA_DIR / "meta_train_predictions_eda6.csv"
)

ants_similarity = pd.read_csv(
    DATA_DIR
    / "meta_train_predictions_antspyx_on_rotation_by_similarity.csv"
)

profiles = pd.read_csv(
    DATA_DIR / "meta_train_predictions_profiles.csv"
)

ants_native = pd.read_csv(
    DATA_DIR / "meta_ants_predictions_from_report.csv"
)

In [2]:
frames = {
    "eda6": eda6,
    "ants_similarity": ants_similarity,
    "profiles": profiles,
    "ants_native": ants_native,
}

for name, frame in frames.items():
    frame["uid"] = frame["uid"].astype(str)

    if frame["uid"].duplicated().any():
        raise ValueError(
            f"Powtórzone UID w pliku: {name}"
        )

    print(
        name,
        frame.shape,
        list(frame.columns),
    )

eda6 (1089, 7) ['uid', 'y_true', 'p_eda6_fold_1', 'p_eda6_fold_2', 'p_eda6_fold_3', 'p_eda6_fold_4', 'p_eda6_fold_5']
ants_similarity (1089, 7) ['uid', 'y_true', 'p_antspyx_fold_1', 'p_antspyx_fold_2', 'p_antspyx_fold_3', 'p_antspyx_fold_4', 'p_antspyx_fold_5']
profiles (1089, 3) ['uid', 'y_true', 'p_profiles']
ants_native (1362, 2) ['uid', 'is_pathologic']


In [3]:
ants_native = ants_native.rename(
    columns={
        "is_pathologic": "p_ants_native_mean"
    }
)

In [4]:
meta_train = (
    eda6
    .merge(
        ants_similarity,
        on=["uid", "y_true"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        profiles,
        on=["uid", "y_true"],
        how="inner",
        validate="one_to_one",
    )
    .merge(
        ants_native[
            ["uid", "p_ants_native_mean"]
        ],
        on="uid",
        how="left",
        validate="one_to_one",
    )
)

if len(meta_train) != len(eda6):
    raise ValueError(
        f"Po połączeniu pozostało {len(meta_train)} "
        f"z {len(eda6)} rekordów."
    )

if meta_train.isna().any().any():
    print(
        meta_train.isna().sum()[
            meta_train.isna().sum() > 0
        ]
    )

    raise ValueError(
        "Po połączeniu występują brakujące wartości."
    )

print("Rozmiar:", meta_train.shape)
display(meta_train.head())

Rozmiar: (1089, 14)


,uid,y_true,p_eda6_fold_1,p_eda6_fold_2,p_eda6_fold_3,p_eda6_fold_4,p_eda6_fold_5,p_antspyx_fold_1,p_antspyx_fold_2,p_antspyx_fold_3,p_antspyx_fold_4,p_antspyx_fold_5,p_profiles,p_ants_native_mean
0,xjn6fokz,0,0.153959,0.285420,0.213447,0.333463,0.266485,0.206324,0.044938,0.593849,0.170358,0.631159,0.180457,0.977207
1,po6j0m3l,1,0.945385,0.831022,0.973613,0.934644,0.973652,0.594830,0.494980,0.708146,0.860590,0.795568,0.775497,0.841055
2,45bg34iq,0,0.509882,0.632679,0.220572,0.405176,0.359730,0.457912,0.233320,0.771167,0.474506,0.576462,0.742272,0.536357
3,g5rifwr0,1,0.997282,0.994022,0.994718,0.978875,0.982774,0.576289,0.739176,0.907781,0.928097,0.962708,0.491510,0.971988
4,6e07fpy1,0,0.103359,0.166019,0.097250,0.091975,0.107103,0.061857,0.006627,0.183021,0.014706,0.087207,0.140297,0.976219


In [5]:
feature_columns = [
    column
    for column in meta_train.columns
    if column.startswith("p_")
]

print("Liczba cech:", len(feature_columns))

for column in feature_columns:
    print(column)

Liczba cech: 12
p_eda6_fold_1
p_eda6_fold_2
p_eda6_fold_3
p_eda6_fold_4
p_eda6_fold_5
p_antspyx_fold_1
p_antspyx_fold_2
p_antspyx_fold_3
p_antspyx_fold_4
p_antspyx_fold_5
p_profiles
p_ants_native_mean


In [6]:
X_meta = meta_train[
    feature_columns
].to_numpy(dtype=np.float32)

y_meta = meta_train[
    "y_true"
].to_numpy(dtype=int)

print(X_meta.shape)
print(np.unique(y_meta, return_counts=True))

(1089, 12)
(array([0, 1]), array([492, 597]))


In [7]:
meta_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression(
        max_iter=5000,
        random_state=42,
    ),
)

parameter_grid = {
    "logisticregression__C": [
        0.001,
        0.003,
        0.01,
        0.03,
        0.1,
        0.3,
        1.0,
    ],
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

meta_search = GridSearchCV(
    estimator=meta_pipeline,
    param_grid=parameter_grid,
    scoring="neg_log_loss",
    cv=cv,
    n_jobs=-1,
    refit=True,
)

meta_search.fit(
    X_meta,
    y_meta,
)

print("Najlepsze parametry:", meta_search.best_params_)
print("CV log loss:", -meta_search.best_score_)

meta_model = meta_search.best_estimator_

Najlepsze parametry: {'logisticregression__C': 0.3}
CV log loss: 0.2328506201505661


In [8]:
train_meta_probs = meta_model.predict_proba(
    X_meta
)[:, 1]

train_meta_probs = np.clip(
    train_meta_probs,
    0.001,
    0.999,
)

train_meta_classes = (
    train_meta_probs >= 0.5
).astype(int)

print(
    "Train log loss:",
    log_loss(y_meta, train_meta_probs),
)

print(
    "Train AUROC:",
    roc_auc_score(y_meta, train_meta_probs),
)

print(
    "Train accuracy:",
    accuracy_score(y_meta, train_meta_classes),
)

print(
    "Train F1:",
    f1_score(y_meta, train_meta_classes),
)

Train log loss: 0.21825793385505676
Train AUROC: 0.9699445738175975
Train accuracy: 0.9100091827364555
Train F1: 0.9166666666666666


In [9]:
META_MODEL_PATH = (
    DATA_DIR / "2final_meta_classifier.joblib"
)

joblib.dump(
    {
        "model": meta_model,
        "feature_columns": feature_columns,
    },
    META_MODEL_PATH,
)

print("Zapisano:", META_MODEL_PATH)

Zapisano: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons\src\notebooks\EDA6\2final_meta_classifier.joblib
